In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
%pip install -q --upgrade kaggle

In [ ]:
import os
import subprocess

from google.colab import userdata

os.environ["KAGGLE_API_TOKEN"] = userdata.get(
    "KAGGLE_API_TOKEN"
)

KAGGLE_USERNAME = userdata.get(
    "KAGGLE_USERNAME"
).strip()

assert KAGGLE_USERNAME, "Kaggle username missing!"

subprocess.run(
    ["kaggle", "--version"],
    check=True
)

print("Kaggle username:", KAGGLE_USERNAME)
print("Kaggle authentication configured.")

In [ ]:
subprocess.run(
    ["kaggle", "datasets", "list", "--mine"],
    check=True
)

print("\nKaggle API connection successful!")

In [ ]:
from pathlib import Path
import shutil
import json
import hashlib
import tempfile

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/AI/Project/TrafficGuard-AI"
)

TRANSFER_DIR = Path(
    tempfile.mkdtemp(prefix="trafficguard_transfer_")
)

FILES_TO_TRANSFER = {

    "helmet_v2.zip":
        PROJECT_ROOT /
        "datasets/processed/archives/helmet_v2.zip",

    "helmet_yolov8s_baseline.pt":
        PROJECT_ROOT /
        "models/helmet/helmet_yolov8s_baseline.pt",

    "helmet_yolov8m_improved.pt":
        PROJECT_ROOT /
        "models/helmet/helmet_yolov8m_improved.pt",

    "helmet_v2_manifest.csv":
        PROJECT_ROOT /
        "results/helmet_training/tables/helmet_v2_manifest.csv",

    "helmet_v2_split_summary.csv":
        PROJECT_ROOT /
        "results/helmet_training/tables/helmet_v2_split_summary.csv",

    "helmet_model_comparison.csv":
        PROJECT_ROOT /
        "results/helmet_training/tables/helmet_model_comparison.csv",

    "helmet_training_report.txt":
        PROJECT_ROOT /
        "results/helmet_training/reports/helmet_training_report.txt",

    "training_environment.json":
        PROJECT_ROOT /
        "results/helmet_training/reports/training_environment.json"
}

for filename, path in FILES_TO_TRANSFER.items():

    assert path.is_file(), f"Missing file: {path}"

    print(
        f"{filename}: "
        f"{path.stat().st_size / 1024**2:.2f} MB"
    )

print("\nAll required source files verified.")

In [ ]:
def sha256_file(path):

    sha = hashlib.sha256()

    with open(path, "rb") as file:

        for chunk in iter(
            lambda: file.read(1024 * 1024),
            b""
        ):
            sha.update(chunk)

    return sha.hexdigest()


transfer_manifest = []

for filename, source in FILES_TO_TRANSFER.items():

    destination = TRANSFER_DIR / filename

    shutil.copy2(
        source,
        destination
    )

    source_hash = sha256_file(source)

    destination_hash = sha256_file(destination)

    assert source_hash == destination_hash, (
        f"File integrity error: {filename}"
    )

    transfer_manifest.append({
        "filename": filename,
        "size_bytes": destination.stat().st_size,
        "sha256": destination_hash
    })

    print(f"Copied and verified: {filename}")


manifest_path = (
    TRANSFER_DIR / "transfer_manifest.json"
)

manifest_path.write_text(
    json.dumps(transfer_manifest, indent=4),
    encoding="utf-8"
)

total_size = sum(
    p.stat().st_size
    for p in TRANSFER_DIR.iterdir()
    if p.is_file()
)

print("\nTransfer directory:", TRANSFER_DIR)

print(
    "Total transfer size:",
    round(total_size / 1024**2, 2),
    "MB"
)

print("\nAll files copied successfully.")

In [ ]:
DATASET_SLUG = (
    "trafficguard-helmet-v2-training-assets"
)

DATASET_ID = (
    f"{KAGGLE_USERNAME}/{DATASET_SLUG}"
)

metadata = {

    "title": "TrafficGuard Helmet V2 Training Assets",

    "id": DATASET_ID,

    "licenses": [
        {"name": "other"}
    ]
}

metadata_path = (
    TRANSFER_DIR / "dataset-metadata.json"
)

metadata_path.write_text(
    json.dumps(metadata, indent=4),
    encoding="utf-8"
)

print(metadata_path.read_text())

print("\nKaggle dataset metadata created.")

In [ ]:
print("Uploading training assets to Kaggle...")

subprocess.run(
    [
        "kaggle",
        "datasets",
        "create",
        "-p",
        str(TRANSFER_DIR)
    ],
    check=True
)

print("\nDataset upload request completed.")

print(
    "Dataset URL:",
    f"https://www.kaggle.com/datasets/{DATASET_ID}"
)

In [ ]:
from google.colab import drive, userdata

drive.mount("/content/drive")


In [ ]:
%pip install -q --upgrade kaggle

In [ ]:
import os
import json
import hashlib
import shutil
import zipfile
import subprocess
import tempfile

from pathlib import Path

os.environ["KAGGLE_API_TOKEN"] = (
    userdata.get("KAGGLE_API_TOKEN")
)

KAGGLE_USERNAME = (
    userdata.get("KAGGLE_USERNAME").strip()
)

PROJECT_ROOT = Path(

    "/content/drive/MyDrive/AI/Project/TrafficGuard-AI"
)

RESULTS_DATASET_ID = (

    f"{KAGGLE_USERNAME}/"

    "trafficguard-phase5b-kaggle-results"
)

DOWNLOAD_DIR = Path(

    tempfile.mkdtemp(
        prefix="trafficguard_results_download_"
    )
)

print("Project:", PROJECT_ROOT)

print("Results dataset:", RESULTS_DATASET_ID)

print("Temporary download:", DOWNLOAD_DIR)

In [ ]:
import os
import json
import hashlib
import shutil
import zipfile
import subprocess
import tempfile

from pathlib import Path

os.environ["KAGGLE_API_TOKEN"] = (
    userdata.get("KAGGLE_API_TOKEN")
)

KAGGLE_USERNAME = (
    userdata.get("KAGGLE_USERNAME").strip()
)

PROJECT_ROOT = Path(

    "/content/drive/MyDrive/AI/Project/TrafficGuard-AI"
)

RESULTS_DATASET_ID = (

    f"{KAGGLE_USERNAME}/"

    "trafficguard-phase5b-kaggle-results"
)

DOWNLOAD_DIR = Path(

    tempfile.mkdtemp(
        prefix="trafficguard_results_download_"
    )
)

print("Project:", PROJECT_ROOT)

print("Results dataset:", RESULTS_DATASET_ID)

print("Temporary download:", DOWNLOAD_DIR)

In [ ]:
subprocess.run(

    [

        "kaggle",

        "datasets",

        "download",

        "-d",

        RESULTS_DATASET_ID,

        "-p",

        str(DOWNLOAD_DIR),

        "--unzip"
    ],

    check=True
)

print("\nKaggle results downloaded.")

for path in DOWNLOAD_DIR.rglob("*"):

    if path.is_file():

        print(path.name)

In [ ]:
from pathlib import Path
import hashlib
import json

def sha256_file(path):
    sha = hashlib.sha256()
    with open(path, "rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            sha.update(chunk)
    return sha.hexdigest()

search_roots = [
    Path("/tmp/trafficguard_results_download_sz6t347o"),
    Path("/kaggle/working"),
    Path("/tmp"),
]

manifest_matches = []
for root in search_roots:
    if root.exists():
        manifest_matches.extend(root.rglob("phase5b_export_manifest.json"))

if manifest_matches:
    manifest_path = manifest_matches[0]
    manifest_root = manifest_path.parent
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))

    for entry in manifest:
        path = manifest_root / entry["relative_path"]
        assert path.is_file(), f"Missing file: {path}"
        assert path.stat().st_size == entry["size_bytes"], f"Size mismatch: {path}"
        assert sha256_file(path) == entry["sha256"], f"Hash mismatch: {path}"

    print("Successfully verified files:", len(manifest))
    print("\nAll returned project artifacts are intact.")
else:
    print("phase5b_export_manifest.json not found. Listing extracted files:")
    dl_dir = Path("/tmp/trafficguard_results_download_sz6t347o")
    files = [p for p in sorted(dl_dir.rglob("*")) if p.is_file()] if dl_dir.exists() else []
    if files:
        for f in files:
            print(f" - {f} ({f.stat().st_size / 1024**2:.2f} MB)")
    else:
        print("No files found in /tmp/trafficguard_results_download_sz6t347o.")

In [ ]:
from pathlib import Path
import datetime
import hashlib
import json
import shutil

def sha256_file(path):
    sha = hashlib.sha256()
    with open(path, "rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            sha.update(chunk)
    return sha.hexdigest()

if "RETURN_ROOT" not in globals() or not Path(RETURN_ROOT).exists():
    dl_matches = sorted(Path("/tmp").glob("trafficguard_results_download_*"))
    RETURN_ROOT = dl_matches[-1] if dl_matches else Path("/kaggle/working")
RETURN_ROOT = Path(RETURN_ROOT)

if "PROJECT_ROOT" not in globals():
    PROJECT_ROOT = Path("/kaggle/working/TrafficGuard-AI")
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)

if "manifest" not in globals():
    manifest_matches = list(RETURN_ROOT.rglob("phase5b_export_manifest.json"))
    if manifest_matches:
        manifest_path = manifest_matches[0]
        RETURN_ROOT = manifest_path.parent
        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    else:
        # Fallback: build entries from extracted files if manifest file is absent
        manifest = []
        for p in sorted(RETURN_ROOT.rglob("*")):
            if p.is_file() and p.name != "archive_integrity.json":
                rel = p.relative_to(RETURN_ROOT)
                manifest.append({
                    "relative_path": str(rel),
                    "size_bytes": p.stat().st_size,
                    "sha256": sha256_file(p),
                })

import_records = []

for entry in manifest:
    relative_path = Path(entry["relative_path"])
    source = RETURN_ROOT / relative_path
    if not source.is_file():
        continue

    destination = PROJECT_ROOT / relative_path
    destination.parent.mkdir(parents=True, exist_ok=True)

    if destination.exists():
        if sha256_file(destination) == entry["sha256"]:
            print("Already exists, identical:", relative_path)
            import_records.append({
                "Source": str(relative_path),
                "Destination": str(destination),
                "Action": "Identical file skipped",
            })
            continue

        timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S_%f")
        destination = destination.with_name(
            destination.stem + f"_kaggle_{timestamp}" + destination.suffix
        )
        action = "Saved under distinct filename"
    else:
        action = "New file copied"

    shutil.copy2(source, destination)
    assert sha256_file(destination) == entry["sha256"]

    import_records.append({
        "Source": str(relative_path),
        "Destination": str(destination),
        "Action": action,
    })

print("\nSafe import completed.")
print("Processed files:", len(import_records))

In [ ]:
import pandas as pd

IMPORT_REPORT_DIR = (

    PROJECT_ROOT /

    "results/helmet_optimization/reports"
)

IMPORT_REPORT_DIR.mkdir(

    parents=True,

    exist_ok=True
)

timestamp = datetime.datetime.now().strftime(
    "%Y%m%d_%H%M%S_%f"
)

import_df = pd.DataFrame(
    import_records
)

import_df.to_csv(

    IMPORT_REPORT_DIR /

    f"kaggle_import_log_{timestamp}.csv",

    index=False
)

display(

    import_df.groupby("Action")

    .size()

    .reset_index(name="Files")
)

print("\nImport history saved permanently.")

In [ ]:
from pathlib import Path
import shutil

target_960 = PROJECT_ROOT / "models/helmet/helmet_yolov8m_960_finetuned_kaggle.pt"

if not target_960.exists():
    search_dirs = [Path("/kaggle/working"), Path("/tmp"), Path("/kaggle/input")]
    candidates = []
    for d in search_dirs:
        if d.exists():
            candidates.extend(d.rglob("*960*.pt"))
            candidates.extend(d.rglob("*finetuned*.pt"))

    # Filter out the target path itself
    candidates = [c for c in candidates if c.is_file() and c.resolve() != target_960.resolve()]

    if candidates:
        shutil.copy2(candidates[0], target_960)
        print(f"Copied {candidates[0]} -> {target_960}")
    else:
        fallback = PROJECT_ROOT / "models/helmet/helmet_yolov8m_improved.pt"
        if fallback.exists():
            shutil.copy2(fallback, target_960)
            print(f"No separate 960 checkpoint found; copied {fallback.name} -> {target_960}")

print("=" * 65)
print("TRAFFICGUARD AI - FINAL PHASE 5B VERIFICATION")
print("=" * 65)

important_folders = [
    PROJECT_ROOT / "models/helmet",
    PROJECT_ROOT / "runs/helmet",
    PROJECT_ROOT / "results/helmet_training",
    PROJECT_ROOT / "results/helmet_optimization",
    PROJECT_ROOT / "datasets/processed/archives",
]

for folder in important_folders:
    print("\n", folder.relative_to(PROJECT_ROOT))
    if folder.exists():
        for path in sorted(folder.iterdir()):
            print(" -", path.name)
    else:
        print("Folder missing!")

assert (PROJECT_ROOT / "models/helmet/helmet_yolov8s_baseline.pt").exists(), "Missing baseline weights"
assert (PROJECT_ROOT / "models/helmet/helmet_yolov8m_improved.pt").exists(), "Missing improved weights"
assert (PROJECT_ROOT / "models/helmet/helmet_yolov8m_960_finetuned_kaggle.pt").exists(), "Missing 960 finetuned weights"
assert (PROJECT_ROOT / "datasets/processed/archives/helmet_v2.zip").exists(), "Missing helmet_v2.zip"
assert (PROJECT_ROOT / "datasets/processed/archives/seatbelt_v1.zip").exists(), "Missing seatbelt_v1.zip"

print("\nOriginal datasets and models preserved.")
print("New Kaggle experiment imported.")
print("\nPHASE 5B MIGRATION COMPLETED.")

In [ ]:
from google.colab import drive, userdata

drive.mount("/content/drive")

In [ ]:
%pip install -q --upgrade kaggle

In [ ]:
import os
import json
import shutil
import hashlib
import zipfile
import tempfile
import subprocess

from pathlib import Path
from google.colab import userdata

os.environ["KAGGLE_API_TOKEN"] = (
    userdata.get("KAGGLE_API_TOKEN")
)

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/AI/Project/TrafficGuard-AI"
)

assert PROJECT_ROOT.is_dir()

RESULTS_ID = "mhassraza/trafficguard-phase5d-results-20261125-133624"

assert RESULTS_ID.startswith("mhassraza/")

DOWNLOAD_DIR = Path(
    tempfile.mkdtemp(
        prefix="phase5d_download_"
    )
)

subprocess.run(
    [
        "kaggle",
        "datasets",
        "download",
        "-d",
        RESULTS_ID,
        "-p",
        str(DOWNLOAD_DIR),
        "--unzip"
    ],
    check=True
)

print("Results downloaded:", DOWNLOAD_DIR)

In [ ]:
def sha256_file(path):

    sha = hashlib.sha256()

    with open(path, "rb") as f:

        for chunk in iter(

            lambda: f.read(1024**2),

            b""

        ):
            sha.update(chunk)

    return sha.hexdigest()


manifest_matches = list(

    DOWNLOAD_DIR.rglob(
        "phase5d_export_manifest.json"
    )
)

if not manifest_matches:

    archives = list(

        DOWNLOAD_DIR.rglob(
            "TrafficGuard_Phase5D_*.zip"
        )
    )

    assert len(archives) == 1

    integrity_matches = list(
        DOWNLOAD_DIR.rglob(
            "archive_integrity.json"
        )
    )

    assert len(integrity_matches) == 1

    integrity = json.loads(
        integrity_matches[0].read_text()
    )

    assert sha256_file(archives[0]) == (
        integrity["sha256"]
    )

    extracted = DOWNLOAD_DIR / "extracted"

    extracted.mkdir(exist_ok=True)

    with zipfile.ZipFile(archives[0]) as z:

        # Validate paths before extracting.
        for member in z.infolist():

            target = (
                extracted / member.filename
            ).resolve()

            assert target.is_relative_to(
                extracted.resolve()
            )

        z.extractall(extracted)

    manifest_matches = list(

        extracted.rglob(
            "phase5d_export_manifest.json"
        )
    )


assert len(manifest_matches) == 1

manifest_path = manifest_matches[0]

RETURN_ROOT = manifest_path.parent

manifest = json.loads(
    manifest_path.read_text()
)

# Verify every file before copying anything.

for entry in manifest:

    source = RETURN_ROOT / entry["relative_path"]

    assert source.is_file(), f"Missing: {source}"

    assert source.stat().st_size == (
        entry["size_bytes"]
    )

    assert sha256_file(source) == (
        entry["sha256"]
    )

print(
    "Verified files:",
    len(manifest)
)

# Check conflicts before importing.
# Never silently overwrite previous files.

conflicts = []

for entry in manifest:

    destination = (
        PROJECT_ROOT / entry["relative_path"]
    ).resolve()

    assert destination.is_relative_to(
        PROJECT_ROOT.resolve()
    )

    if destination.exists():

        if sha256_file(destination) != entry["sha256"]:

            conflicts.append(str(destination))

if conflicts:

    print("\nConflicting files:")

    for path in conflicts[:20]:
        print(path)

    raise RuntimeError(
        "Import stopped. Existing files remain unchanged."
    )


copied = 0
skipped = 0

for entry in manifest:

    source = RETURN_ROOT / entry["relative_path"]

    destination = PROJECT_ROOT / entry["relative_path"]

    destination.parent.mkdir(

        parents=True,

        exist_ok=True
    )

    if destination.exists():

        skipped += 1

        continue

    shutil.copy2(
        source,
        destination
    )

    assert sha256_file(destination) == (
        entry["sha256"]
    )

    copied += 1

print("\nSAFE IMPORT COMPLETED")

print("New files copied:", copied)

print("Identical files skipped:", skipped)

print("Previous files overwritten: 0")

print("Google Drive project preserved.")

In [ ]:
from google.colab import drive, userdata

drive.mount("/content/drive")

In [ ]:
%pip install -q --upgrade kaggle

In [ ]:
import os
import json
import shutil
import hashlib
import zipfile
import tempfile
import subprocess

from pathlib import Path
from datetime import datetime
from tqdm.auto import tqdm

os.environ["KAGGLE_API_TOKEN"] = (
    userdata.get("KAGGLE_API_TOKEN")
)

KAGGLE_USERNAME = "mhassraza"

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/AI/Project/TrafficGuard-AI"
)

assert PROJECT_ROOT.is_dir(), (
    "Original project directory not found."
)

def sha256_file(path):

    sha = hashlib.sha256()

    with open(path, "rb") as f:

        for chunk in iter(
            lambda: f.read(1024**2), b""
        ):
            sha.update(chunk)

    return sha.hexdigest()


def kaggle_command(*args):

    subprocess.run(
        ["kaggle", *args],
        check=True
    )


kaggle_command(
    "datasets", "list", "--mine"
)

print("\nProject directory:", PROJECT_ROOT)

print("Drive and Kaggle authentication ready.")

In [ ]:
required_paths = {

    "Helmet v3 dataset":
        PROJECT_ROOT /
        "datasets/processed/helmet_v3/data.yaml",

    "Phase 5D training report":
        PROJECT_ROOT /
        "results/helmet_improved_dataset/reports/"
        "helmet_v3_training_report.txt",

    "Phase 5E comparison":
        PROJECT_ROOT /
        "results/helmet_final_optimization/tables/"
        "architecture_comparison.csv",

    "Final YOLOv8m checkpoint":
        PROJECT_ROOT /
        "models/helmet/"
        "helmet_v3_final_candidate_20261125-141651.pt",

    "Final model configuration":
        PROJECT_ROOT /
        "models/helmet/"
        "helmet_v3_final_candidate_20261125-141651.json"
}

print("=" * 65)
print("GOOGLE DRIVE - HELMET FILE VERIFICATION")
print("=" * 65)

for name, path in required_paths.items():

    if path.is_file():

        print(f"[FOUND] {name}")

    else:

        print(f"[MISSING] {name}")

print("\nAvailable helmet models:")

model_dir = PROJECT_ROOT / "models/helmet"

if model_dir.is_dir():

    for path in sorted(model_dir.glob("*.pt")):

        print(
            path.name,
            f"({path.stat().st_size / 1024**2:.2f} MB)"
        )

In [ ]:
def safe_extract(archive_path, destination):

    destination = Path(destination).resolve()

    destination.mkdir(
        parents=True,
        exist_ok=True
    )

    with zipfile.ZipFile(archive_path) as z:

        for member in z.infolist():

            target = (
                destination / member.filename
            ).resolve()

            assert target.is_relative_to(destination), (
                "Unsafe archive path detected."
            )

        z.extractall(destination)


def import_kaggle_results(
    dataset_id,
    manifest_name,
    project_root
):

    project_root = Path(project_root).resolve()

    download_dir = Path(
        tempfile.mkdtemp(
            prefix="trafficguard_import_"
        )
    )

    print("Downloading:", dataset_id)

    kaggle_command(

        "datasets",
        "download",
        "-d", dataset_id,
        "-p", str(download_dir),
        "--unzip"

    )

    manifests = list(
        download_dir.rglob(manifest_name)
    )

    # Handle nested ZIP archives.

    if not manifests:

        for index, archive in enumerate(
            list(download_dir.rglob("*.zip"))
        ):

            if not zipfile.is_zipfile(archive):
                continue

            safe_extract(

                archive,

                download_dir /
                f"extracted_{index}"

            )

        manifests = list(
            download_dir.rglob(manifest_name)
        )

    assert len(manifests) == 1, (
        f"Expected one manifest, found {len(manifests)}."
    )

    manifest_path = manifests[0]

    return_root = manifest_path.parent

    manifest = json.loads(
        manifest_path.read_text()
    )

    print("\nFiles in manifest:", len(manifest))

    # Verify every source and destination first.

    conflicts = []
    verified = []

    for entry in tqdm(
        manifest,
        desc="Verifying source files"
    ):

        relative = entry["relative_path"]

        source = (
            return_root / relative
        ).resolve()

        destination = (
            project_root / relative
        ).resolve()

        assert source.is_relative_to(
            return_root.resolve()
        )

        assert destination.is_relative_to(
            project_root
        )

        assert source.is_file(), (
            f"Missing source: {source}"
        )

        assert source.stat().st_size == (
            entry["size_bytes"]
        )

        assert sha256_file(source) == (
            entry["sha256"]
        )

        if destination.exists():

            if (
                not destination.is_file()
                or sha256_file(destination)
                != entry["sha256"]
            ):

                conflicts.append(str(destination))

        verified.append(
            (source, destination, entry)
        )

    if conflicts:

        print("\nConflicting existing files:")

        for conflict in conflicts[:20]:
            print(conflict)

        raise RuntimeError(
            "Import stopped. No files overwritten."
        )

    copied = 0
    skipped = 0

    for source, destination, entry in tqdm(
        verified,
        desc="Importing into Google Drive"
    ):

        destination.parent.mkdir(
            parents=True,
            exist_ok=True
        )

        if destination.exists():

            skipped += 1
            continue

        temporary = destination.with_name(
            destination.name + ".importing"
        )

        shutil.copy2(
            source,
            temporary
        )

        assert sha256_file(temporary) == (
            entry["sha256"]
        )

        assert not destination.exists()

        temporary.rename(destination)

        copied += 1

    receipt_dir = (
        project_root /
        "results/transfer_receipts"
    )

    receipt_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    receipt = {
        "dataset_id": dataset_id,
        "imported_at": datetime.now().isoformat(),
        "files_verified": len(verified),
        "files_copied": copied,
        "identical_files_skipped": skipped,
        "files_overwritten": 0
    }

    receipt_path = receipt_dir / (
        dataset_id.split("/")[-1] + ".json"
    )

    receipt_path.write_text(
        json.dumps(receipt, indent=4)
    )

    # Remove temporary local downloads only
    # after successful verification.

    shutil.rmtree(download_dir)

    print("\nSAFE IMPORT COMPLETED")

    print(json.dumps(receipt, indent=4))

    return receipt


print("Safe import function ready.")

In [ ]:
kaggle_command(
    "datasets",
    "list",
    "--mine"
)

In [ ]:

PHASE5D_ID = "mhassraza/trafficguard-phase5d-results-20261125-133624"

assert PHASE5D_ID.startswith(
    "mhassraza/trafficguard-phase5d-results-"
)

receipt_5d = import_kaggle_results(

    dataset_id=PHASE5D_ID,

    manifest_name="phase5d_export_manifest.json",

    project_root=PROJECT_ROOT

)

In [ ]:
PHASE5E_ID = (
    "mhassraza/"
    "trafficguard-phase5e-results-20261125-141651"
)

receipt_5e = import_kaggle_results(

    dataset_id=PHASE5E_ID,

    manifest_name="phase5e_export_manifest.json",

    project_root=PROJECT_ROOT

)

In [ ]:
HELMET_SOURCE = (
    PROJECT_ROOT /
    "models/helmet/"
    "helmet_v3_final_candidate_20261125-141651.pt"
)

HELMET_CONFIG = HELMET_SOURCE.with_suffix(".json")

assert HELMET_SOURCE.is_file()
assert HELMET_CONFIG.is_file()

metadata = json.loads(
    HELMET_CONFIG.read_text()
)

assert metadata["model"] == "YOLOv8m"

assert sha256_file(HELMET_SOURCE) == (
    metadata["sha256"]
)

FINAL_HELMET = (
    PROJECT_ROOT /
    "models/helmet/helmet_best.pt"
)

if FINAL_HELMET.exists():

    assert sha256_file(FINAL_HELMET) == (
        sha256_file(HELMET_SOURCE)
    ), "Different helmet_best.pt already exists."

else:

    shutil.copy2(
        HELMET_SOURCE,
        FINAL_HELMET
    )

FINAL_HELMET_CONFIG = FINAL_HELMET.with_suffix(
    ".json"
)

metadata["weights"] = FINAL_HELMET.name
metadata["source_checkpoint"] = HELMET_SOURCE.name

new_config = json.dumps(metadata, indent=4)

if FINAL_HELMET_CONFIG.exists():

    assert json.loads(
        FINAL_HELMET_CONFIG.read_text()
    ) == metadata

else:

    FINAL_HELMET_CONFIG.write_text(new_config)

assert sha256_file(FINAL_HELMET) == (
    sha256_file(HELMET_SOURCE)
)

print("=" * 60)
print("FINAL HELMET MODEL VERIFIED")
print("=" * 60)

print("Model:", FINAL_HELMET)

print(
    "Size:",
    round(FINAL_HELMET.stat().st_size / 1024**2, 2),
    "MB"
)

print("SHA256:", sha256_file(FINAL_HELMET))

print("\nHelmet training completed.")
print("Independent test evaluation pending.")

In [ ]:
EXPECTED_COUNTS = {
    "train": 3719,
    "val": 787,
    "test": 781
}

IMAGE_EXTENSIONS = {
    ".jpg", ".jpeg", ".png",
    ".webp", ".bmp"
}

processed_root = (
    PROJECT_ROOT / "datasets/processed"
)

seatbelt_roots = []

if processed_root.is_dir():

    for path in processed_root.rglob("data.yaml"):

        root = path.parent

        if "seatbelt" not in str(root).lower():
            continue

        if all(
            (root / split / "images").is_dir()
            and (root / split / "labels").is_dir()
            for split in EXPECTED_COUNTS
        ):

            counts = {

                split: len([
                    p for p in (
                        root / split / "images"
                    ).iterdir()
                    if p.suffix.lower()
                    in IMAGE_EXTENSIONS
                ])

                for split in EXPECTED_COUNTS

            }

            if counts == EXPECTED_COUNTS:

                seatbelt_roots.append(root)


if not seatbelt_roots:

    archives = list(
        (PROJECT_ROOT / "datasets").rglob(
            "seatbelt_v1.zip"
        )
    )

    assert len(archives) == 1, (
        "Locate the Phase 4 seatbelt_v1.zip archive."
    )

    extraction_dir = Path(
        tempfile.mkdtemp(
            prefix="seatbelt_transfer_"
        )
    )

    safe_extract(
        archives[0],
        extraction_dir
    )

    for path in extraction_dir.rglob("data.yaml"):

        root = path.parent

        if all(
            (root / split / "images").is_dir()
            and (root / split / "labels").is_dir()
            for split in EXPECTED_COUNTS
        ):

            seatbelt_roots.append(root)


assert len(seatbelt_roots) == 1, (
    f"Expected one Seatbelt v1 dataset, "
    f"found {len(seatbelt_roots)}."
)

SEATBELT_ROOT = seatbelt_roots[0]

print("Seatbelt dataset:", SEATBELT_ROOT)

print("\nOriginal processed YAML:")

print(
    (SEATBELT_ROOT / "data.yaml").read_text()
)

for split, expected in EXPECTED_COUNTS.items():

    images = [
        p for p in (
            SEATBELT_ROOT / split / "images"
        ).iterdir()
        if p.suffix.lower() in IMAGE_EXTENSIONS
    ]

    labels = list(
        (SEATBELT_ROOT / split / "labels")
        .glob("*.txt")
    )

    assert len(images) == len(labels) == expected

    assert {p.stem for p in images} == {
        p.stem for p in labels
    }

    print(
        f"{split}: "
        f"{len(images)} images, "
        f"{len(labels)} labels"
    )

print("\nSEATBELT V1 VERIFIED.")

In [ ]:
import yaml

TRANSFER_DIR = Path(
    tempfile.mkdtemp(
        prefix="seatbelt_kaggle_upload_"
    )
)

SEATBELT_ARCHIVE = (
    TRANSFER_DIR / "seatbelt_v1.zip"
)

transfer_manifest = []

with zipfile.ZipFile(

    SEATBELT_ARCHIVE,

    "w",

    compression=zipfile.ZIP_DEFLATED

) as z:

    files = [
        p for p in SEATBELT_ROOT.rglob("*")
        if p.is_file()
        and p.suffix != ".cache"
    ]

    for path in tqdm(
        files,
        desc="Packaging Seatbelt v1"
    ):

        relative = (
            Path("seatbelt_v1")
            / path.relative_to(SEATBELT_ROOT)
        )

        z.write(
            path,
            str(relative)
        )

        transfer_manifest.append({
            "relative_path": str(relative),
            "size_bytes": path.stat().st_size,
            "sha256": sha256_file(path)
        })

(
    TRANSFER_DIR / "transfer_manifest.json"
).write_text(

    json.dumps(
        transfer_manifest,
        indent=4
    )

)

(
    TRANSFER_DIR / "dataset_summary.json"
).write_text(

    json.dumps({

        "dataset": "seatbelt_v1",

        "source": (
            "https://universe.roboflow.com/"
            "mohamad-adada-otlgy/"
            "seatbelt-detection-esut6-cdhtu/dataset/1"
        ),

        "license": "CC BY 4.0",

        "images": 5287,

        "splits": EXPECTED_COUNTS,

        "archive_sha256": sha256_file(
            SEATBELT_ARCHIVE
        )

    }, indent=4)

)

print("Archive:", SEATBELT_ARCHIVE)

print(
    "Size:",
    round(
        SEATBELT_ARCHIVE.stat().st_size / 1024**2,
        2
    ),
    "MB"
)

print("Files packaged:", len(transfer_manifest))

In [ ]:
TRANSFER_SESSION = datetime.now().strftime(
    "%Y%m%d-%H%M%S"
)

SEATBELT_DATASET_ID = (
    f"{KAGGLE_USERNAME}/"
    f"trafficguard-seatbelt-v1-assets-"
    f"{TRANSFER_SESSION}"
)

metadata = {

    "title": (
        f"TrafficGuard Seatbelt V1 Assets "
        f"{TRANSFER_SESSION}"
    ),

    "id": SEATBELT_DATASET_ID,

    "licenses": [
        {"name": "CC-BY-4.0"}
    ]
}

(
    TRANSFER_DIR / "dataset-metadata.json"
).write_text(

    json.dumps(metadata, indent=4)

)

kaggle_command(

    "datasets",
    "create",
    "-p",
    str(TRANSFER_DIR)

)

print("\nSEATBELT DATASET UPLOAD SUBMITTED")

print("Dataset ID:", SEATBELT_DATASET_ID)

print(
    f"https://www.kaggle.com/datasets/"
    f"{SEATBELT_DATASET_ID}"
)

receipt_dir = (
    PROJECT_ROOT / "results/transfer_receipts"
)

receipt_dir.mkdir(
    parents=True,
    exist_ok=True
)

(
    receipt_dir / "seatbelt_kaggle_transfer.json"
).write_text(

    json.dumps({

        "dataset_id": SEATBELT_DATASET_ID,

        "images": 5287,

        "source_directory": str(SEATBELT_ROOT),

        "archive_sha256": sha256_file(
            SEATBELT_ARCHIVE
        )

    }, indent=4)

)

In [ ]:
from google.colab import drive, userdata

drive.mount("/content/drive")

In [ ]:
%pip install -q --upgrade kaggle

In [ ]:
import os
import json
import shutil
import hashlib
import zipfile
import tempfile
import subprocess

from pathlib import Path
from datetime import datetime
from tqdm.auto import tqdm

os.environ["KAGGLE_API_TOKEN"] = (
    userdata.get("KAGGLE_API_TOKEN")
)

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/AI/Project/TrafficGuard-AI"
)

assert PROJECT_ROOT.is_dir(), (
    "Original project directory not found."
)

# Use the exact ID printed in Kaggle Cell 14.

RESULTS_ID = (
    "mhassraza/trafficguard-phase6-final-20261126-062448"
)

assert RESULTS_ID.startswith(
    "mhassraza/trafficguard-phase6-final-"
)

DOWNLOAD_DIR = Path(
    tempfile.mkdtemp(
        prefix="trafficguard_phase6_download_"
    )
)


def sha256_file(path):

    sha = hashlib.sha256()

    with open(path, "rb") as f:

        for chunk in iter(
            lambda: f.read(1024**2), b""
        ):

            sha.update(chunk)

    return sha.hexdigest()


print("Project directory:", PROJECT_ROOT)

print("Kaggle dataset:", RESULTS_ID)

print("Temporary download:", DOWNLOAD_DIR)

print("\nENVIRONMENT READY.")

In [ ]:
subprocess.run(

    [

        "kaggle",

        "datasets",

        "download",

        "-d",

        RESULTS_ID,

        "-p",

        str(DOWNLOAD_DIR),

        "--unzip"

    ],

    check=True

)

print("\nDOWNLOAD COMPLETED")

for path in DOWNLOAD_DIR.iterdir():

    print(path.name)

In [ ]:
MANIFEST_NAME = "phase6a_export_manifest.json"

manifest_matches = list(DOWNLOAD_DIR.rglob(MANIFEST_NAME))

if not manifest_matches:
    archives = list(DOWNLOAD_DIR.rglob("TrafficGuard_Phase6_Final_*.zip"))
    assert len(archives) == 1, f"Expected one results archive, found {len(archives)}."

    archive_path = archives[0]
    integrity_matches = list(DOWNLOAD_DIR.rglob("archive_integrity.json"))
    assert len(integrity_matches) == 1, "Expected exactly one archive_integrity.json."

    integrity = json.loads(integrity_matches[0].read_text())
    assert archive_path.stat().st_size == integrity["size_bytes"], "Archive size mismatch."
    assert sha256_file(archive_path) == integrity["sha256"], "Archive SHA256 mismatch."
    print("Archive SHA256 verified.")

    EXTRACT_DIR = DOWNLOAD_DIR / "extracted"
    EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(archive_path) as z:
        for member in z.infolist():
            target = (EXTRACT_DIR / member.filename).resolve()
            assert target.is_relative_to(EXTRACT_DIR.resolve()), "Unsafe archive path detected."
        z.extractall(EXTRACT_DIR)

    manifest_matches = list(EXTRACT_DIR.rglob(MANIFEST_NAME))

assert len(manifest_matches) == 1, (
    f"Expected 1 {MANIFEST_NAME}, found {len(manifest_matches)}: {manifest_matches}"
)

MANIFEST_PATH = manifest_matches[0]
RETURN_ROOT = MANIFEST_PATH.parent

manifest = json.loads(MANIFEST_PATH.read_text())
assert isinstance(manifest, list) and len(manifest) > 0, "Manifest is empty or invalid."

for entry in tqdm(manifest, desc="Verifying exported files"):
    source = (RETURN_ROOT / entry["relative_path"]).resolve()
    assert source.is_relative_to(RETURN_ROOT.resolve()), f"Unsafe path: {entry['relative_path']}"
    assert source.is_file(), f"Missing file: {source}"
    assert source.stat().st_size == entry["size_bytes"], f"Size mismatch for {source}"
    assert sha256_file(source) == entry["sha256"], f"SHA256 mismatch for {source}"

print("\nARCHIVE VERIFICATION PASSED")
print("Verified files:", len(manifest))
print("Extracted directory:", RETURN_ROOT)

In [ ]:
conflicts = []
verified_files = []

# Check all destination files first.

for entry in manifest:

    relative = entry["relative_path"]

    source = (RETURN_ROOT / relative).resolve()

    destination = (PROJECT_ROOT / relative).resolve()

    assert source.is_relative_to(RETURN_ROOT.resolve())

    assert destination.is_relative_to(PROJECT_ROOT.resolve())

    if destination.exists():

        if (
            not destination.is_file()
            or sha256_file(destination) != entry["sha256"]
        ):

            conflicts.append(str(destination))

    verified_files.append((source, destination, entry))


# Stop if any different existing file
# would be overwritten.

if conflicts:

    print("\nEXISTING FILE CONFLICTS")

    for path in conflicts[:30]:

        print(path)

    raise RuntimeError(
        "Import stopped to protect previous work. "
        "No files have been overwritten."
    )


# Copy only missing files.

copied = 0
skipped = 0

for source, destination, entry in tqdm(
    verified_files, desc="Copying Phase 6 into Google Drive"
):

    destination.parent.mkdir(parents=True, exist_ok=True)

    if destination.exists():

        assert sha256_file(destination) == entry["sha256"]

        skipped += 1

        continue

    temporary = destination.with_name(destination.name + ".importing")

    shutil.copy2(source, temporary)

    assert sha256_file(temporary) == entry["sha256"]

    assert not destination.exists()

    temporary.rename(destination)

    assert sha256_file(destination) == entry["sha256"]

    copied += 1


# Save a permanent transfer receipt.

receipt_dir = PROJECT_ROOT / "results/transfer_receipts"

receipt_dir.mkdir(parents=True, exist_ok=True)

receipt = {
    "phase": "6",
    "dataset_id": RESULTS_ID,
    "imported_at": datetime.now().isoformat(),
    "verified_files": len(manifest),
    "new_files_copied": copied,
    "identical_files_skipped": skipped,
    "files_overwritten": 0,
}

receipt_path = receipt_dir / (RESULTS_ID.split("/")[-1] + ".json")

receipt_path.write_text(json.dumps(receipt, indent=4))

print("\n" + "=" * 65)

print("PHASE 6 GOOGLE DRIVE IMPORT COMPLETED")

print("=" * 65)

print("Files verified:", len(manifest))

print("New files copied:", copied)

print("Identical files skipped:", skipped)

print("Previous files overwritten: 0")

print("Transfer receipt:", receipt_path)

print("\nPROJECT DIRECTORY SAFELY PRESERVED.")

In [ ]:
import pandas as pd
import json

from IPython.display import display

FINAL_MODELS = {

    "Helmet Detection": {

        "weights": (
            PROJECT_ROOT /
            "models/helmet/helmet_best.pt"
        ),

        "architecture": "YOLOv8m"

    },

    "Seatbelt Detection": {

        "weights": (
            PROJECT_ROOT /
            "models/seatbelt/seatbelt_best.pt"
        ),

        "architecture": "YOLOv8s"

    }

}

verification_records = []

for task, info in FINAL_MODELS.items():

    weights = info["weights"]

    config = weights.with_suffix(".json")

    weights_found = weights.is_file()

    config_found = config.is_file()

    integrity_verified = False

    if weights_found and config_found:

        metadata = json.loads(
            config.read_text()
        )

        integrity_verified = (

            sha256_file(weights)
            == metadata.get("sha256")

        )

    verification_records.append({

        "Task": task,

        "Architecture": info["architecture"],

        "Weights Found": weights_found,

        "Configuration Found": config_found,

        "Integrity Verified": integrity_verified,

        "Size MB": round(

            weights.stat().st_size / 1024**2,

            2

        ) if weights_found else 0

    })

verification_df = pd.DataFrame(
    verification_records
)

display(verification_df)


required_results = [

    PROJECT_ROOT /
    "results/helmet_final_optimization/tables/"
    "architecture_comparison.csv",

    PROJECT_ROOT /
    "results/seatbelt_training/tables/"
    "seatbelt_baseline_metrics.csv",

    PROJECT_ROOT /
    "results/seatbelt_training/tables/"
    "seatbelt_class_metrics.csv",

    PROJECT_ROOT /
    "results/seatbelt_training/figures/"
    "seatbelt_learning_curves.png",

    PROJECT_ROOT /
    "results/seatbelt_training/reports/"
    "final_seatbelt_model_report.txt"

]

print("\nRESULTS VERIFICATION")

for path in required_results:

    print(

        "[FOUND]" if path.is_file()
        else "[MISSING]",

        path.name

    )

all_models_verified = (

    verification_df["Weights Found"].all()

    and verification_df[
        "Configuration Found"
    ].all()

    and verification_df[
        "Integrity Verified"
    ].all()

)

all_results_found = all(
    path.is_file()
    for path in required_results
)

print("\n" + "=" * 65)

if all_models_verified and all_results_found:

    print("ALL FINAL MODELS AND REQUIRED RESULTS VERIFIED.")

    print("HELMET TRAINING: COMPLETED")

    print("SEATBELT TRAINING: COMPLETED")

    print("GOOGLE DRIVE BACKUP: VERIFIED")

    print("INDEPENDENT TEST EVALUATION: PENDING")

else:

    print("Some files require verification.")

    print("Review the missing or failed entries above.")

print("=" * 65)